# Python 코드 독해와 LLM 활용 배경

RAG 실습 코드는 대부분 익숙한 파이썬 문법 위에 만들어집니다.  
문법 때문에 코드 흐름을 놓치지 않도록, 문서 처리와 LLM 예제에서 반복되는 코드 모양을 정리합니다.


## 핵심 코드 패턴

- 노트북 셀을 위에서 아래로 실행할 수 있습니다.
- 문자열, 리스트, 딕셔너리, 함수, 반복문을 읽을 수 있습니다.
- `obj.value`, `obj.method(...)`, `metadata.get(...)` 같은 객체 사용 문법을 이해합니다.
- 리스트 안의 딕셔너리, 딕셔너리 안의 딕셔너리처럼 중첩된 데이터를 읽을 수 있습니다.
- `enumerate`, `zip`, 리스트 컴프리헨션, `set`, `sorted`, `lambda`를 코드 독해 관점에서 이해합니다.
- ChatGPT 같은 대중형 LLM을 기업 업무에 그대로 쓰기 어려운 이유를 설명할 수 있습니다.


## LLM 기본 용어

아래 용어는 깊은 이론보다 의미 구분이 중요합니다.

- LLM: 텍스트를 입력받아 답변을 생성하는 언어 모델
- 프롬프트: 모델에게 보내는 지시문과 입력
- 문서 기반 답변: 내부 문서나 매뉴얼을 근거로 답변하게 만드는 접근
- 검증: 답변이 근거와 맞는지 확인하는 과정

세부 구현보다 입력, 문서, 답변, 검증이 어떤 역할을 하는지 구분하는 데 집중합니다.


# 1부. 노트북 실행과 값 표현


## 1. 셀 실행 순서

노트북은 앞 셀에서 만든 값을 뒤 셀이 이어서 사용합니다.  
오류가 나면 먼저 필요한 앞 셀을 실행했는지 확인합니다.


In [ ]:
course_name = 'RAG 기초 과정'
day_count = 3

print(course_name)
print(f'{day_count}일 과정입니다.')


In [ ]:
message = f'{course_name}에서는 문서를 활용한 LLM 응답을 다룹니다.'
message


## 2. 문자열과 f-string

질문, 문서 본문, 프롬프트는 문자열로 다루는 경우가 많습니다.  
f-string은 문자열 안에 변수 값을 넣을 때 사용합니다.


In [ ]:
question = '회의실 예약은 어떻게 하나요?'
page = 12

line = f'[page {page}] {question}'
print(line)


긴 문자열은 따옴표 세 개로 만들 수 있습니다. 여러 줄 지시문이나 문서 조각을 만들 때 자주 쓰입니다.


In [ ]:
template = f'''
아래 질문에 답하세요.

질문: {question}
'''

print(template)


## 3. import 읽기

`import`는 파이썬 기본 기능이나 외부 라이브러리의 기능을 불러오는 문법입니다.  
무엇을 불러오는지부터 확인합니다.


In [ ]:
import textwrap
from pathlib import Path

long_text = '회의실 예약은 사내 포털에서 신청하며, 참석 인원과 사용 시간을 입력해야 합니다.'
print(textwrap.shorten(long_text, width=36, placeholder=' ...'))

data_path = Path('data') / 'sample.pdf'
print(data_path)


# 2부. 리스트와 딕셔너리 데이터


## 4. 리스트와 슬라이싱

리스트는 여러 값을 순서대로 담습니다.  
검색 결과, 문서 목록, 질문 목록을 다룰 때 사용합니다.


In [ ]:
queries = [
    '회의실 예약 방법은?',
    '휴가 신청은 언제까지 해야 하나요?',
    '공지사항은 어디에서 확인하나요?',
]

print('전체 개수:', len(queries))
print('첫 번째:', queries[0])
print('앞의 두 개:', queries[:2])


## 5. 딕셔너리와 `.get()`

딕셔너리는 이름표가 붙은 값을 담습니다.  
문서의 페이지 번호, 출처, ID처럼 부가 정보를 담을 때 사용합니다.


In [ ]:
metadata = {
    'source': 'office_guide.pdf',
    'page': 12,
    'doc_id': 'guide-012',
}

print(metadata['source'])
print(metadata.get('page'))
print(metadata.get('section', 'section unknown'))


`.get()`은 해당 키가 없을 때 오류를 내지 않고 기본값을 돌려줍니다.  
문서마다 메타데이터 구성이 조금씩 다를 수 있어 자주 쓰입니다.


## 6. 중첩된 데이터 읽기

문서 처리 코드에는 리스트 안에 딕셔너리가 들어 있는 구조가 자주 나옵니다.  
아래 예제는 여러 질문과 기준 답변을 묶은 형태입니다.


In [ ]:
question_rows = [
    {
        'query': '회의실 예약 방법은?',
        'answer': '사내 포털에서 회의실과 시간을 선택해 신청한다.',
        'source_pages': [12, 13],
    },
    {
        'query': '휴가 신청은 언제까지 해야 하나요?',
        'answer': '휴가 시작 최소 3일 전까지 신청한다.',
        'source_pages': [20],
    },
]

print(question_rows[0]['query'])
print(question_rows[0]['source_pages'][0])


# 3부. 반복문과 데이터 변환


## 7. for와 enumerate

`for`는 목록을 하나씩 처리합니다.  
`enumerate`는 순번이 필요할 때 사용합니다.


In [ ]:
for row in question_rows:
    print(row['query'])

print('---')

for idx, row in enumerate(question_rows, start=1):
    print(f'{idx}. {row["query"]}')


## 8. zip으로 두 목록 함께 보기

`zip`은 두 목록을 나란히 묶어 처리할 때 사용합니다.


In [ ]:
doc_texts = [
    '회의실 예약은 사내 포털에서 신청한다.',
    '참석 인원과 사용 시간을 입력한다.',
]
doc_pages = [12, 13]

for text, page in zip(doc_texts, doc_pages):
    print(f'p.{page}: {text}')


## 9. 리스트 컴프리헨션

리스트 컴프리헨션은 리스트를 짧게 만드는 문법입니다.  
처음에는 아래 두 코드가 같은 일을 한다고 보면 됩니다.


In [ ]:
queries_a = []
for row in question_rows:
    queries_a.append(row['query'])

queries_b = [row['query'] for row in question_rows]

print(queries_a)
print(queries_b)


조건을 붙이면 필요한 값만 고를 수 있습니다.


In [ ]:
multi_page_rows = [row for row in question_rows if len(row['source_pages']) >= 2]
multi_page_rows


## 10. 문자열 합치기와 줄바꿈 정리

여러 문서 조각을 하나의 문자열로 합치는 코드는 문서 기반 답변 예제에서 자주 나옵니다.  
`join`, `replace`, `strip`을 많이 사용합니다.


In [ ]:
raw_texts = [
    '회의실 예약은\n사내 포털에서 신청한다. ',
    ' 참석 인원과 사용 시간을 입력한다.',
]

cleaned_texts = [text.replace('\n', ' ').strip() for text in raw_texts]
combined_text = '\n\n'.join(cleaned_texts)

print(combined_text)


## 11. set과 중복 제거

`set`은 중복 없는 값 묶음입니다.  
여러 결과를 합칠 때 이미 본 ID를 기록하는 용도로 사용합니다.


In [ ]:
doc_ids = ['guide-012', 'guide-013', 'guide-012', 'guide-020']

seen = set()
unique_ids = []

for doc_id in doc_ids:
    if doc_id in seen:
        continue
    seen.add(doc_id)
    unique_ids.append(doc_id)

print(unique_ids)


## 12. sorted와 lambda

점수나 길이를 기준으로 정렬하는 코드도 자주 나옵니다.  
`lambda item: item['score']`는 정렬 기준으로 `score` 값을 쓰겠다는 뜻입니다.


In [ ]:
results = [
    {'text': '회의실 예약 안내', 'score': 0.82},
    {'text': '휴가 신청 방법', 'score': 0.41},
    {'text': '공지 확인 방법', 'score': 0.91},
]

ranked = sorted(results, key=lambda item: item['score'], reverse=True)

for item in ranked:
    print(item['score'], item['text'])


# 4부. 함수와 객체 문법


## 13. 함수의 입력, 기본값, return

함수는 입력을 받아 결과를 돌려주는 코드 묶음입니다.  
출력 미리보기, 문서 문자열 정리, 결과 변환은 함수로 묶어 두면 읽기 쉽습니다.


In [ ]:
def preview_text(text, width=40):
    cleaned = text.replace('\n', ' ').strip()
    return textwrap.shorten(cleaned, width=width, placeholder=' ...')

print(preview_text(raw_texts[0]))
print(preview_text(raw_texts[0], width=24))


타입 힌트가 붙은 함수도 나옵니다. 실행 방식이 바뀌는 것은 아니고, 입력과 출력의 예상 타입을 읽기 쉽게 적어 둔 것입니다.


In [ ]:
def normalize_query(query: str) -> str:
    return query.strip().replace('?', '')

normalize_query('  회의실 예약은 어떻게 하나요?  ')


## 14. 객체의 속성과 메서드

라이브러리 객체는 보통 값과 기능을 함께 갖습니다.

- `obj.value`: 객체 안의 값 읽기
- `obj.method(...)`: 객체가 가진 기능 실행

아래 예제는 문서 객체의 느낌을 보기 위한 간단한 클래스입니다.


In [ ]:
from dataclasses import dataclass, field

@dataclass
class DemoDocument:
    page_content: str
    metadata: dict = field(default_factory=dict)

    def preview(self, width=36):
        return preview_text(self.page_content, width=width)

doc = DemoDocument(
    page_content='회의실 예약은 사내 포털에서 신청한다.',
    metadata={'page': 12, 'doc_id': 'guide-012'},
)

print(doc.page_content)
print(doc.metadata.get('page'))
print(doc.preview())


## 15. 객체 목록 다루기

문서 객체 여러 개를 리스트로 묶어서 처리하는 코드가 자주 나옵니다.


In [ ]:
docs = [
    DemoDocument('회의실 예약은 사내 포털에서 신청한다.', {'page': 12, 'doc_id': 'guide-012'}),
    DemoDocument('참석 인원과 사용 시간을 입력한다.', {'page': 13, 'doc_id': 'guide-013'}),
    DemoDocument('휴가 신청은 시작일 최소 3일 전까지 등록한다.', {'page': 20, 'doc_id': 'guide-020'}),
]

for idx, doc in enumerate(docs, start=1):
    page = doc.metadata.get('page', 'N/A')
    print(f'{idx}. p.{page} | {doc.preview()}')


## 16. 키워드 인자

함수나 객체를 호출할 때 `name=value` 형태로 값을 넘기는 문법을 키워드 인자라고 합니다.  
`k=4`, `temperature=0`, `width=80` 같은 형태는 설정값을 이름으로 넘기는 방식입니다.


In [ ]:
def search_by_keyword(docs, keyword, k=2):
    hits = [doc for doc in docs if keyword in doc.page_content]
    return hits[:k]

hits = search_by_keyword(docs, keyword='확인', k=1)
for doc in hits:
    print(doc.page_content)


## 17. 튜플 언패킹

두 값을 한 묶음으로 돌려받고 나눠 담는 코드도 자주 나옵니다.


In [ ]:
scored_docs = [
    (docs[0], 0.82),
    (docs[1], 0.91),
]

for doc, score in scored_docs:
    print(score, doc.preview())


# 5부. 기업 LLM 활용 배경


## 18. ChatGPT 같은 대중형 LLM

ChatGPT 같은 서비스는 사람이 브라우저에서 직접 질문하고 답을 받기 좋습니다.  
요약, 번역, 글쓰기, 코드 설명, 아이디어 정리처럼 개인 생산성 작업에 빠르게 활용할 수 있습니다.

<img src="image/chatgpt.jpg" width="520">

다만 기업 업무에서는 사용 방식이 달라집니다.  
편리함만 보고 내부 정보를 그대로 입력하면 보안, 기록, 책임 소재 문제가 생길 수 있습니다.


## 19. 기업에서 보는 선택지

<img src="image/enterprise_llm_options.svg" width="820">

| 선택지 | 장점 | 주의점 |
|---|---|---|
| 대중형 챗봇 | 접근이 쉽고 빠르게 써볼 수 있음 | 민감 정보 입력과 결과 추적 관리가 어려울 수 있음 |
| API 기반 상용 LLM | 앱, 서버, 업무 시스템에 연결하기 좋음 | 비용, 데이터 처리 정책, 장애 대응을 확인해야 함 |
| 오픈소스 LLM | 내부망 배포와 직접 제어가 가능 | 작은 모델은 성능이 부족할 수 있고 튜닝, GPU, 운영 역량이 필요 |

오픈소스 LLM은 보안 통제 면에서 매력적일 수 있지만, 항상 상용 대형 모델 수준의 품질이 바로 나오는 것은 아닙니다.  
모델 성능, 추론 속도, GPU 비용, 운영 인력을 함께 봐야 합니다.


## 20. 기업 업무에서 조심할 점

<img src="image/llm_enterprise_risks.svg" width="820">

| 구분 | 확인할 점 |
|---|---|
| 입력 데이터 | 고객 정보, 상담 기록, 내부 문서, 계약 정보가 포함되는가 |
| 저장과 학습 | 입력한 데이터가 어디에 저장되고 학습에 쓰이는가 |
| 권한 | 사용자가 봐도 되는 문서만 답변에 쓰이는가 |
| 추적 | 누가 어떤 질문을 했고 어떤 답을 받았는지 확인 가능한가 |
| 책임 | 틀린 답변이 업무 판단에 쓰였을 때 어떻게 검토할 것인가 |


## 21. 할루시네이션

LLM은 모르는 내용도 자연스러운 문장으로 답할 수 있습니다.  
이처럼 근거가 부족한 내용을 그럴듯하게 생성하는 현상을 할루시네이션이라고 부릅니다.

| 상황 | 리스크 |
|---|---|
| 내부 문서를 제공하지 않음 | 회사 절차와 다른 답변을 할 수 있음 |
| 최신 변경 사항이 없음 | 폐기된 절차나 오래된 정보를 말할 수 있음 |
| 숫자, 코드, 조항 번호 질문 | 비슷하지만 틀린 값을 만들 수 있음 |
| 출처 확인이 없음 | 답변을 검증하기 어려움 |

업무에서는 답변이 그럴듯한지보다 근거가 있는지가 더 중요합니다.


## 22. 내부 문서 기반 답변의 과제

기업 문서 기반 LLM 활용에서는 아래 질문이 중요합니다.

- LLM이 모르는 내부 문서를 어떻게 참고하게 할 수 있을까?
- 긴 문서 중 질문과 관련 있는 부분을 어떻게 찾을까?
- 답변이 문서 근거를 벗어나지 않게 하려면 어떤 구조가 필요할까?
- 만든 시스템이 잘 동작하는지 어떻게 확인할까?

이 질문들은 문서 검색, 답변 생성, 검증 구조로 이어집니다.


## 23. 문법과 LLM 리스크 확인

아래 질문에 짧게 답해 봅니다.

1. 노트북에서 앞 셀을 실행하지 않으면 뒤 셀에서 왜 오류가 날 수 있나요?
2. `metadata.get('page', 'N/A')`처럼 `.get()`을 쓰는 이유는 무엇인가요?
3. 리스트 안의 딕셔너리에서 첫 번째 질문의 `query`를 꺼내려면 어떤 식으로 접근하나요?
4. `enumerate(items, start=1)`는 언제 사용하나요?
5. 리스트 컴프리헨션은 어떤 코드를 짧게 쓴 형태인가요?
6. `set()`은 어떤 상황에서 유용한가요?
7. `sorted(..., key=lambda item: item['score'], reverse=True)`는 어떤 기준으로 정렬하나요?
8. `obj.value`와 `obj.method()`는 각각 무엇을 의미하나요?
9. ChatGPT 같은 대중형 LLM을 기업 업무에 그대로 쓰기 어려운 이유는 무엇인가요?
10. 할루시네이션이 업무에서 위험한 이유는 무엇인가요?


## 예시 답변

1. 앞 셀에서 만든 변수나 함수가 뒤 셀에서 사용될 수 있기 때문입니다.
2. 키가 없을 때 오류를 내지 않고 기본값을 사용하기 위해서입니다.
3. `rows[0]['query']`처럼 리스트 인덱스와 딕셔너리 키를 차례로 사용합니다.
4. 반복하면서 순번도 함께 출력하거나 저장해야 할 때 사용합니다.
5. `for`로 값을 하나씩 꺼내 새 리스트에 `append`하는 코드를 짧게 쓴 형태입니다.
6. 중복을 제거하거나 이미 본 값을 기록할 때 유용합니다.
7. `score` 값이 큰 항목이 앞에 오도록 내림차순 정렬합니다.
8. `obj.value`는 객체 안의 값을 읽는 것이고, `obj.method()`는 객체가 가진 기능을 실행하는 것입니다.
9. 민감 정보 입력, 데이터 저장 정책, 권한 관리, 결과 추적, 책임 소재 문제가 생길 수 있기 때문입니다.
10. 자연스러워 보이지만 실제 근거가 없는 답변이 업무 판단에 사용될 수 있기 때문입니다.
